# Проект: Предсказание стоимости жилья

**Описание проекта**:

В данном проекте необходимо обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. На основе указанных данных нужно предсказать медианную стоимость дома в жилом массиве — `'median_house_value'`.

**Задачи проекта**:
- обучить две модели линейной регресии (одна - со всеми признаками, другая - только с числовыми);
- сделать предсказания на тестовой выборке;
- оценить качество моделей с помощью метрик RMSE, MAE и R2.

## Описание данных

В колонках датасета содержатся следующие данные:

- `'longitude'` — широта;
- `'latitude'` — долгота;
- `'housing_median_age'` — медианный возраст жителей жилого массива;
- `'total_rooms'` — общее количество комнат в домах жилого массива;
- `'total_bedrooms'` — общее количество спален в домах жилого массива;
- `'population'` — количество человек, которые проживают в жилом массиве;
- `'households'` — количество домовладений в жилом массиве;
- `'median_income'` — медианный доход жителей жилого массива;
- `'median_house_value'` — медианная стоимость дома в жилом массиве, целевой признак;
- `'ocean_proximity'` — близость к океану.

## Загрузка библиотек и инициализация сессии

Перед тем, как приступить к задаче, нужно подготовить необходимые инструменты.

In [ ]:
import pandas as pd
import numpy as np

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F
# from pyspark.sql import Window

# работа с пайплайнами
from pyspark.ml import Pipeline
# кросс-валидация и перебор параметров
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# заполнение пропусков
from pyspark.ml.feature import Imputer
# работа с признаками (1 - с категориальными, 2 и 3 - с числовыми)
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
# модель
from pyspark.ml.regression import LinearRegression
# метрики
from pyspark.ml.evaluation import RegressionEvaluator

# работа с признаками (с категориальными)
pyspark_version = pyspark.__version__
if int(pyspark_version[:1]) == 3:
    from pyspark.ml.feature import OneHotEncoder as OHE
elif int(pyspark_version[:1]) == 2:
    from pyspark.ml.feature import OneHotEncodeEstimator as OHE

In [ ]:
RANDOM_SEED = 2022 # задаем константу

In [ ]:
# инициализируем сессию
spark = SparkSession.builder \
                    .master("local") \
                    .appName("ML California Housing") \
                    .getOrCreate()

## Загрузка данных

Теперь загрузим наш датасет и взглянем на него.

In [ ]:
#загрузка
df_housing = spark.read.load('/datasets/housing.csv',
                             format="csv", sep=",", inferSchema=True, header="true")

In [ ]:
# предварительный просмотр
df_housing.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



Исходя из описания данных теоретически мы бы могли встретить следующие типы данных:
- `StringType` — строковый тип данных, который используется для хранения текстовых значений.
- `DoubleType` — двойной тип данных с плавающей точкой, который используется для представления чисел с дробной частью и является более точным, чем FloatType.
- `FloatType` — тип данных с одинарной точностью с плавающей точкой. Аналогичен DoubleType, но имеет меньшую точность.
- `IntegerType` — целочисленный тип данных, который редставляет целые числа без дробной части.

Как мы видем после изучения схемы, все столбцы кроме `'ocean_proximity'` (тип содержимого которого StringType) имеют тип DoubleType. Это в целом соответствует содержимому, однако некоторым столбцам (`'total_rooms'`, `'total_bedrooms'`, `'population'`, `'households'`) все-таки лучше подошел бы тип IntegerType, ведь указанные параметры никак не могут быть дробными.

Так же мы видим, что во всех столбцах могут быть пропуски.

Взглянем на само содержимое датасета.

In [ ]:
# краткое описание
df_housing.describe().toPandas()

,summary,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,count,20640,20640,20640,20640,20433,20640,20640,20640,20640,20640
1,mean,-119.56970445736148,35.6318614341087,28.639486434108527,2635.7630813953488,537.8705525375618,1425.4767441860465,499.5396802325581,3.8706710029070246,206855.81690891474,None
2,stddev,2.003531723502584,2.135952397457101,12.58555761211163,2181.6152515827944,421.38507007403115,1132.46212176534,382.3297528316098,1.899821717945263,115395.61587441359,None
3,min,-124.35,32.54,1.0,2.0,1.0,3.0,1.0,0.4999,14999.0,<1H OCEAN
4,max,-114.31,41.95,52.0,39320.0,6445.0,35682.0,6082.0,15.0001,500001.0,NEAR OCEAN


In [ ]:
# вывод первых 10 строк в более удобном для восприятия виде
df_housing.limit(10).toPandas()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY
6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY
7,-122.25,37.84,52.0,3104.0,687.0,1157.0,647.0,3.1200,241400.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY
9,-122.25,37.84,52.0,3549.0,707.0,1551.0,714.0,3.6912,261100.0,NEAR BAY


In [ ]:
# выводим первые 10 строк, вариант 2 (вертикальный)
df_housing.show(10, vertical=True)

-RECORD 0----------------------
 longitude          | -122.23  
 latitude           | 37.88    
 housing_median_age | 41.0     
 total_rooms        | 880.0    
 total_bedrooms     | 129.0    
 population         | 322.0    
 households         | 126.0    
 median_income      | 8.3252   
 median_house_value | 452600.0 
 ocean_proximity    | NEAR BAY 
-RECORD 1----------------------
 longitude          | -122.22  
 latitude           | 37.86    
 housing_median_age | 21.0     
 total_rooms        | 7099.0   
 total_bedrooms     | 1106.0   
 population         | 2401.0   
 households         | 1138.0   
 median_income      | 8.3014   
 median_house_value | 358500.0 
 ocean_proximity    | NEAR BAY 
-RECORD 2----------------------
 longitude          | -122.24  
 latitude           | 37.85    
 housing_median_age | 52.0     
 total_rooms        | 1467.0   
 total_bedrooms     | 190.0    
 population         | 496.0    
 households         | 177.0    
 median_income      | 7.2574   
 median_

Также, вероятно тип IntegerType подошел бы и столбцу `'housing_median_age'`, однако мы не можем быть уверены, что где-то в датасете нет нецелых значений в данном столбце.

Также еще раз видим, что все признаки, за исключением близости к морю (который является категориальным), относятся к числовым, но при этом имеют разный порядок единиц измерения. Это значит, что для улучшения качества некоторых моделей числовые признаки будет необходимо масштабировать.

Перейдем к подготовке данных.

## Подготовка данных

### Типы данных

Для начала поменяем типы данных на более подходящие.

In [ ]:
df_housing = df_housing.withColumn('total_rooms', F.col('total_rooms').cast(IntegerType()))

In [ ]:
df_housing = df_housing.withColumn('total_bedrooms', F.col('total_bedrooms').cast(IntegerType()))

In [ ]:
df_housing = df_housing.withColumn('population', F.col('population').cast(IntegerType()))

In [ ]:
df_housing = df_housing.withColumn('households', F.col('households').cast(IntegerType()))

In [ ]:
# проверка
# вывод названия колонок и их типов
print(pd.DataFrame(df_housing.dtypes, columns=['column', 'type']).head(10))

               column    type
0           longitude  double
1            latitude  double
2  housing_median_age  double
3         total_rooms     int
4      total_bedrooms     int
5          population     int
6          households     int
7       median_income  double
8  median_house_value  double
9     ocean_proximity  string


Теперь типы данных лучше соответствуют содержимому.

### Работа с пропусками

Для начала посмотрим, есть ли у нас пропуски в приниципе.

In [ ]:
# проверка пропусков
columns = df_housing.columns

for column in columns:
    print(column, df_housing.where(F.isnan(column) | F.col(column).isNull()).count())

longitude 0
latitude 0
housing_median_age 0
total_rooms 0
total_bedrooms 207
population 0
households 0
median_income 0
median_house_value 0
ocean_proximity 0


Пропуски все-таки есть. И по условиям проекта мы должны их заполнить на свое усмотрение (а вот вариант с удалением там не предусмотрен).

Заполним их медианой (это будет более корректно, особенно с учетом выбросов). Для этих целей применим Imputer. Однако правильно будет сделать это не сейчас, а позже, уже после разбиения на выборки, чтобы не допустить утечки.

А пока перейдем к работе с дубликатами.

### Работа с дубликатами

Удалим дубликаты, если они у нас есть.

In [ ]:
# посмотреть дубликаты
duplicates = (
    df_housing.groupBy(df_housing.columns)
    .count()
    .filter(F.col("count") > 1)
    .drop("count")
)
duplicates.show()
# количество дубликатов
print(f'Количество дубликатов: {duplicates.count()}')

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+



Количество дубликатов: 0


Полные дубликаты не обнаружены.

### Разбиение на выборки и преобразование данных

Так как Импутеры (так же как Энкодеры и Скалеры) нельзя обучать на всех данных, предварительно разобьем наш датасет на две части — выборку для обучения (80%) и выборку для тестирования качества модели (20%).

In [ ]:
# разбиение на выборки
# [.8,.2]: 80% - обучающая, 20% - тестовая
train_data, test_data = df_housing.randomSplit([.8,.2], seed=RANDOM_SEED)
print(train_data.count(), test_data.count())

16418 4222


А также разделим колонки на два типа: числовые и текстовые, которые представляют категориальные данные. И обозначим целевой признак.

In [ ]:
# категориальный признак
categorical_cols = ['ocean_proximity']

In [ ]:
# числовые признаки
numerical_cols = ['longitude', 'latitude', 'housing_median_age',
            'total_rooms', 'total_bedrooms',
            'population', 'households', 'median_income']

In [ ]:
# целевой признак
target = 'median_house_value'

#### Заполнение пропусков

Для работы с пропусками самый полезный инструмент - это Импутер, к тому же он позволяет корректно работать с выборками. Будем использовать его для заполения пропусков медианой.

In [ ]:
# смотрим пропуски в тренировочной выборке
train_data.select([F.count(F.when(F.isnan(c)
                | F.col(c).isNull(), c)).alias(c) for c in train_data.columns]).show(vertical=True)

-RECORD 0-----------------
 longitude          | 0   
 latitude           | 0   
 housing_median_age | 0   
 total_rooms        | 0   
 total_bedrooms     | 173 
 population         | 0   
 households         | 0   
 median_income      | 0   
 median_house_value | 0   
 ocean_proximity    | 0   



In [ ]:
# смотрим пропуски в тестовой выборке
test_data.select([F.count(F.when(F.isnan(c)
                | F.col(c).isNull(), c)).alias(c) for c in test_data.columns]).show(vertical=True)

-RECORD 0-----------------
 longitude          | 0   
 latitude           | 0   
 housing_median_age | 0   
 total_rooms        | 0   
 total_bedrooms     | 34  
 population         | 0   
 households         | 0   
 median_income      | 0   
 median_house_value | 0   
 ocean_proximity    | 0   



In [ ]:
# инициализируем и тренируем импутер
imputer = Imputer(strategy='median', inputCols=['total_bedrooms'], outputCols=['total_bedrooms']).fit(train_data)

In [ ]:
# заполняем пропуски в тренировочной данных
train_data = imputer.transform(train_data)

In [ ]:
# смотрим пропуски в тренировочной выборке
train_data.select([F.count(F.when(F.isnan(c)
                | F.col(c).isNull(), c)).alias(c) for c in train_data.columns]).show(vertical=True)

-RECORD 0-----------------
 longitude          | 0   
 latitude           | 0   
 housing_median_age | 0   
 total_rooms        | 0   
 total_bedrooms     | 0   
 population         | 0   
 households         | 0   
 median_income      | 0   
 median_house_value | 0   
 ocean_proximity    | 0   



In [ ]:
# заполняем пропуски в тестовых данных
test_data = imputer.transform(test_data)

In [ ]:
# смотрим пропуски в тестовой выборке
test_data.select([F.count(F.when(F.isnan(c)
                | F.col(c).isNull(), c)).alias(c) for c in test_data.columns]).show(vertical=True)

-RECORD 0-----------------
 longitude          | 0   
 latitude           | 0   
 housing_median_age | 0   
 total_rooms        | 0   
 total_bedrooms     | 0   
 population         | 0   
 households         | 0   
 median_income      | 0   
 median_house_value | 0   
 ocean_proximity    | 0   



#### Категориальные признаки

Трансформируем категориальные признаки.

Начнем с StringIndexer.Он переводит текстовые категории в числовое представление, так как большинство ML-алгоритмов работает с числовыми данными. По умолчанию трансформер на вход принимает названия колонок, которые нужно трансформировать, и список названий новых колонок.

In [ ]:
# трансформация категориальных признаков с помощью StringIndexer
# инициализируем и обучаем
indexer = StringIndexer(inputCols=categorical_cols,
                        outputCols=[c+'_idx' for c in categorical_cols],
                        handleInvalid = 'keep').fit(train_data)

In [ ]:
# преобразуем тренировочные данные
train_data = indexer.transform(train_data)

cols_tr = [c for c in train_data.columns for i in categorical_cols if (c.startswith(i))]

In [ ]:
# преобразуем тестовые данные
test_data = indexer.transform(test_data)

cols_t = [c for c in test_data.columns for i in categorical_cols if (c.startswith(i))]

Дополнительно можно создать OHE-кодирование для категорий. Оно работает так же, как StringIndexer, — принимает на вход названия колонок, которые нужно трансформировать, и список названий новых колонок.

In [ ]:
# трансформация категориальных признаков с помощью OHE
# инициализируем и обучаем
encoder = OHE(inputCols=[c+'_idx' for c in categorical_cols],
                        outputCols=[c+'_ohe' for c in categorical_cols]).fit(train_data)

In [ ]:
# преобразуем тренировочные данные
train_data = encoder.transform(train_data)

cols_tr = [c for c in train_data.columns for i in categorical_cols if (c.startswith(i))]

In [ ]:
# преобразуем тестовые данные
test_data = encoder.transform(test_data)

cols_t = [c for c in test_data.columns for i in categorical_cols if (c.startswith(i))]

In [ ]:
# проверка преобразований
# тренирвочные данные
train_data.select(cols_tr).show(3)

+---------------+-------------------+-------------------+
|ocean_proximity|ocean_proximity_idx|ocean_proximity_ohe|
+---------------+-------------------+-------------------+
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
+---------------+-------------------+-------------------+
only showing top 3 rows



In [ ]:
# проверка преобразований
# тестовые данные
test_data.select(cols_t).show(3)

+---------------+-------------------+-------------------+
|ocean_proximity|ocean_proximity_idx|ocean_proximity_ohe|
+---------------+-------------------+-------------------+
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
+---------------+-------------------+-------------------+
only showing top 3 rows



Финальный шаг преобразований — это объединение признаков в один вектор, с которым ML-алгоритм умеет работать.

In [ ]:
# объединение признаков в один вектор
# инициализируем
categorical_assembler = \
        VectorAssembler(inputCols=[c+'_ohe' for c in categorical_cols],
                                        outputCol="categorical_features")

In [ ]:
# пребразуем тренировочные данные
train_data = categorical_assembler.transform(train_data)

In [ ]:
# пребразуем тестовые данные
test_data = categorical_assembler.transform(test_data)

Работа с категориальными признаками окончена. Перейдем к числовым.

#### Числовые признаки

Для числовых признаков тоже нужна трансформация — шкалирование значений — чтобы сильные выбросы не смещали предсказания модели.

Возможно, потенциально мы бы могли избавиться от признаков признаков ширина и долгота ввиду вероятной незначительности их влияния на целевой признак. Но в заданах проекта указано, что нужно использавать все признаки, поэтому сохраним их.

In [ ]:
# векторизация числовых признаков
# инициализируем
numerical_assembler = VectorAssembler(inputCols=numerical_cols,
                      outputCol="numerical_features")

In [ ]:
# пребразуем тренировочные данные
train_data = numerical_assembler.transform(train_data)

In [ ]:
# пребразуем тестовые данные
test_data = numerical_assembler.transform(test_data)

In [ ]:
# применение StandardScaler
# инициализируем и тренируем
standardScaler = StandardScaler(inputCol='numerical_features',
                                outputCol="numerical_features_scaled").fit(train_data)

In [ ]:
# пребразуем тренировочные данные
train_data = standardScaler.transform(train_data)

In [ ]:
# пребразуем тестовые данные
test_data = standardScaler.transform(test_data)

Посмотрим, что у нас получилось после всех преобразований.

In [ ]:
# колонки - результат преобразований
# тренировочная выборка
print(train_data.columns)

['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'ocean_proximity_idx', 'ocean_proximity_ohe', 'categorical_features', 'numerical_features', 'numerical_features_scaled']


In [ ]:
# колонки - результат преобразований
# тестовая выборка
print(test_data.columns)

['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'ocean_proximity_idx', 'ocean_proximity_ohe', 'categorical_features', 'numerical_features', 'numerical_features_scaled']


Признаки преобразованы.

#### Собираем признаки

Остался финальный шаг — собрать трансформированные категорийные и числовые признаки с помощью VectorAssembler.

In [ ]:
# собираем признаки
all_features = ['categorical_features','numerical_features_scaled']

final_assembler = VectorAssembler(inputCols=all_features,
                                  outputCol="features")

In [ ]:
# преобразуем тренировочные данные
train_data = final_assembler.transform(train_data)

In [ ]:
# преобразуем тестовые данные
# тренировочная выборка
test_data = final_assembler.transform(test_data)

In [ ]:
# проверка результата
# тренировочная выборка
train_data.select(all_features).show(3)

+--------------------+-------------------------+
|categorical_features|numerical_features_scaled|
+--------------------+-------------------------+
|       (4,[2],[1.0])|     [-61.952887791441...|
|       (4,[2],[1.0])|     [-61.927977100733...|
|       (4,[2],[1.0])|     [-61.913030686308...|
+--------------------+-------------------------+
only showing top 3 rows



In [ ]:
# проверка результата 2
train_data.select("features").show(3)

+--------------------+
|            features|
+--------------------+
|[0.0,0.0,1.0,0.0,...|
|[0.0,0.0,1.0,0.0,...|
|[0.0,0.0,1.0,0.0,...|
+--------------------+
only showing top 3 rows



In [ ]:
# проверка результата
# тестовая выборка
test_data.select(all_features).show(3)

+--------------------+-------------------------+
|categorical_features|numerical_features_scaled|
+--------------------+-------------------------+
|       (4,[2],[1.0])|     [-61.927977100733...|
|       (4,[2],[1.0])|     [-61.893102133741...|
|       (4,[2],[1.0])|     [-61.883137857458...|
+--------------------+-------------------------+
only showing top 3 rows



In [ ]:
# проверка результата 2
# тестовая выборка
test_data.select("features").show(3)

+--------------------+
|            features|
+--------------------+
|[0.0,0.0,1.0,0.0,...|
|[0.0,0.0,1.0,0.0,...|
|[0.0,0.0,1.0,0.0,...|
+--------------------+
only showing top 3 rows



Признаки успешно преобразованы.

## Обучение моделей

Перейдем к непосредсвенной работе с Линейной регресией. Но при инициализации ее необходимо настроить.

В контексте Apache Spark параметры для класса LinearRegression имеют следующее назначение:

- `labelCol (string)` — название столбца в DataFrame, который содержит метки классов (целевые значения). Этот параметр указывает, какой столбец следует использовать в качестве меток классов для модели логистической регрессии.

- `featuresCol (string)` — название столбца, содержащего признаки (независимые переменные). Здесь указывается, какой столбец в DataFrame содержит данные признаков, которые будут использоваться для обучения модели логистической регрессии.

- `maxIter (integer)` — максимальное количество итераций для алгоритма оптимизации. Параметр определяет, сколько раз алгоритм будет пытаться улучшить модель логистической регрессии, прежде чем остановиться. Большее значение maxIter может привести к более точному результату, но также может увеличить время обучения.

- `regParam (double)` — параметр регуляризации (лямбда). Он контролирует степень регуляризации модели, чтобы предотвратить переобучение. Чем больше значение regParam, тем сильнее регуляризация и тем меньше вероятность переобучения модели.

- `elasticNetParam (double)` — коэффициент эластичной сети. Этот параметр определяет, насколько сильно применяется эластичная сеть регуляризации. Эластичная сеть является комбинацией L1 и L2 регуляризации, что помогает предотвратить переобучение модели и улучшить её обобщающую способность.

Первая модель линейной регресии будет использовать все признаки.

In [ ]:
# инициализируем модель
lrF = LinearRegression(labelCol=target, # целевой признак
                         featuresCol='features', # используемые признаки
                         maxIter=10, # максимальное количество попыток, чтобы не уйти в бесконечность
                         regParam=0.3, # регуляризация раз
                         elasticNetParam=0.8) # регуляризация два

In [ ]:
# обучаем модель
modelF = lrF.fit(train_data)

25/01/07 19:59:45 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/01/07 19:59:45 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


Сохраним трансформированную таблицу с колонкой предсказания первой модели в переменной predictions:

In [ ]:
# сохраняем название колонки
predictions_col = 'prediction'

In [ ]:
# получаем предсказание
predictionsF = modelF.transform(test_data)

In [ ]:
# сохраняем предсказания первой модели в отдельную таблицу
# для зрительной оценки
predictedLabesF = predictionsF.select(target, predictions_col)
predictedLabesF.show()

+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|          103600.0|182381.42558303662|
|           50800.0|231958.15269474522|
|           58100.0|158055.72001596307|
|           68400.0|161069.30142661533|
|           72200.0|179694.00335049978|
|           67000.0|170284.80168316327|
|           81300.0|169438.12488770206|
|           70500.0|182212.77405659505|
|           60000.0|159802.82909377804|
|          109400.0|202052.46585005336|
|           74100.0|168073.51044610539|
|           74700.0|187418.53626101883|
|           90000.0|227896.53923914908|
|          104200.0| 219030.4506407592|
|           74100.0|175322.96557153808|
|           67500.0| 166635.5044951376|
|          103100.0|  71047.3646006973|
|           92500.0| 177508.4806094747|
|          128100.0|235160.54384411918|
|           99600.0| 204546.1429572152|
+------------------+------------------+
only showing top 20 rows



Как мы можем видеть - модель не очень точна, мягко говоря. Но насколько - это мы увидим позже, получив ее метрики.

Вторая модель будет аналогична, но уже без категориального признака.

In [ ]:
# инициализируем модель
lrS = LinearRegression(labelCol=target, # целевой признак
                       featuresCol='numerical_features_scaled', # используемые признаки
                       maxIter=10, # максимальное количество попыток, чтобы не уйти в бесконечность
                       regParam=0.3, # регуляризация раз
                       elasticNetParam=0.8) # регуляризация два

In [ ]:
# обучаем модель
modelS = lrS.fit(train_data)

In [ ]:
# получаем предсказание
predictionsS = modelS.transform(test_data)

In [ ]:
# сохраняем предсказания в отдельную таблицу
# для зрительной оценки
predictedLabesS = predictionsS.select(target, predictions_col)
predictedLabesS.show()

+------------------+-------------------+
|median_house_value|         prediction|
+------------------+-------------------+
|          103600.0| 101299.44438905967|
|           50800.0| 185023.28943977505|
|           58100.0| 109506.17933798814|
|           68400.0|   78897.1681606397|
|           72200.0| 129261.46312185284|
|           67000.0| 119502.65214002272|
|           81300.0| 117719.77777141705|
|           70500.0| 133139.48834597832|
|           60000.0| 113263.34172083857|
|          109400.0|  118010.8600577563|
|           74100.0| 121084.34893037332|
|           74700.0| 139638.19276301237|
|           90000.0| 178011.58859190438|
|          104200.0| 168625.02305037342|
|           74100.0| 125749.25233905902|
|           67500.0| 115912.14771372173|
|          103100.0|-23128.385085891932|
|           92500.0| 140081.43080375018|
|          128100.0| 189853.74335895432|
|           99600.0| 153277.54196037631|
+------------------+-------------------+
only showing top

Вторая модель тоже не выглядит высокоточной. А значит, время получить метрики и проанализировать с их помощью результат.

## Анализ результатов

### Метрики

Теперь необходимо оценить наши модели с помощью метрик RMSE, MAE и R². Вспомним, что они из себя представляют:

- `RMSE` (корень среднеквадратичной ошибки) - квадратный корень из суммы квадратов ошибок, разделенной на количество объектов прогноза. Измеряется в тех же единицах, что и целевой признак, показывает, на сколько в среднем ошибается модель. Чем он меньше, тем точнее модель.

- `MAE` (средняя абсолютная ошибка) - рассчитывается как среднее от суммы всех ошибок модели по модулю. Измеряется в тех же единицах, что и целевой признак. Чем MAE меньше, тем лучше модель предсказывает целевой признак. Менее чувствительна к выбросам, чем RMSE.

- `R²` (коэффициент детерминации) - для расчёта нужно разделить сумму квадратов средней ошибки на дисперсию целевого признака. Позволяет сравнивать модели с разными масштабами целевого признака. Оценивает обобщающую способность модели, то есть он определяет, насколько хорошо модель «объясняет», предсказывает целевой признак по входным. Положительные значения R² показывают, в скольких процентах случаев предсказание модели ближе к истине, чем среднее значение целевого признака. Отрицательный R² означает, что средние значения всегда лучше прогнозных. Единица — это его максимальное значение, которое говорит о том, что модель не ошибается.

Оценим качество модели на обучающих данных.

In [ ]:
# оценка первой модели на обучающих данных
trainingSummaryF = modelF.summary
print('Метрики первой модели (со всеми признаками) на тренировочных данных')
print('RMSE: %f' % trainingSummaryF.rootMeanSquaredError)
print('MAE: %f' % trainingSummaryF.meanAbsoluteError)
print('R2: %f' % trainingSummaryF.r2)

Метрики первой модели (со всеми признаками) на тренировочных данных
RMSE: 69339.578489
MAE: 50033.391964
R2: 0.637291


In [ ]:
# оценка второй модели на обучающих данных
trainingSummaryS = modelS.summary
print('Метрики второй модели (только с числовыми признаками) на тренировочных данных')
print('RMSE: %f' % trainingSummaryS.rootMeanSquaredError)
print('MAE: %f' % trainingSummaryS.meanAbsoluteError)
print('R2: %f' % trainingSummaryS.r2)

Метрики второй модели (только с числовыми признаками) на тренировочных данных
RMSE: 69927.946497
MAE: 51014.980985
R2: 0.631109




Теперь проверим на тестовых данных.

In [ ]:
# оценка первой модели на тестовых данных
testSummaryF = modelF.evaluate(test_data)
print('Метрики первой модели (со всеми признаками) на тестовых данных')
print('RMSE: %f' % testSummaryF.rootMeanSquaredError)
print('MAE: %f' % testSummaryF.meanAbsoluteError)
print('R2: %f' % testSummaryF.r2)

Метрики первой модели (со всеми признаками) на тестовых данных
RMSE: 68985.619220
MAE: 50088.731934
R2: 0.648493


In [ ]:
# оценка второй модели на тестовых данных
testSummaryS = modelS.evaluate(test_data)
print('Метрики первой модели (только с числовыми признаками) на тестовых данных')
print('RMSE: %f' % testSummaryS.rootMeanSquaredError)
print('MAE: %f' % testSummaryS.meanAbsoluteError)
print('R2: %f' % testSummaryS.r2)

Метрики первой модели (только с числовыми признаками) на тестовых данных
RMSE: 69179.235365
MAE: 50874.007814
R2: 0.646517


Метрики обеих моделей близки между собой, однако первая модель все-таки предсказывает результаты несколько точнее.

### Анализ важности признаков

Чтобы лучше понять модель линейной регрессии, можно изучить ее коэффициенты и перехват. Эти значения представляют веса, назначенные каждому признаку, и смещение, соответственно.

В контексте Apache Spark, коэффициенты и intercept (перехват) в модели линейной регрессии представляют собой следующие значения:

- `coefficients` — это массив коэффициентов для каждого признака в модели. Он показывает, как каждый признак влияет на прогнозируемое значение целевой переменной.

- `intercept` — это перехват или константа в уравнении линейной регрессии. Она представляет собой смещение модели и показывает, каким будет прогнозируемое значение, если все признаки равны нулю.

Сделаем это для первой модели.

In [ ]:
coefficientsF = modelF.coefficients
interceptF = modelF.intercept

print("Coefficients: ", coefficientsF)
print("Intercept: {:.3f}".format(interceptF))

Coefficients:  [19206.221004658204,-42306.4351890654,28591.465318893243,16008.532760058239,-23616.265925086525,-23383.776404935215,13827.477225283154,-0.0,28567.853446044483,-48099.46640632711,25304.034467213736,71418.51679527076]
Intercept: -997758.499


In [ ]:
# смотрим порядок вариантов категориального признака
indexer.labels

['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY']

In [ ]:
# порядок числовых
numerical_cols

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income']

In [ ]:
# создаем список для будущей важности
featCols = indexer.labels+numerical_cols
featCols

['<1H OCEAN',
 'INLAND',
 'NEAR OCEAN',
 'NEAR BAY',
 'longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income']

In [ ]:
# совмещаем признаки и их важность
feature_importance_f = sorted(list(zip(featCols,
                                       map(abs, coefficientsF))), key=lambda x: x[1], reverse=True)

print("Feature Importance:")
for feature, importance in feature_importance_f:
    print("  {}: {:.3f}".format(feature, importance))

Feature Importance:
  median_income: 71418.517
  population: 48099.466
  INLAND: 42306.435
  NEAR OCEAN: 28591.465
  total_bedrooms: 28567.853
  households: 25304.034
  longitude: 23616.266
  latitude: 23383.776
  <1H OCEAN: 19206.221
  NEAR BAY: 16008.533
  housing_median_age: 13827.477
  total_rooms: 0.000


In [ ]:
#spark.stop()

## Выводы

В рамках данной работы мы:
- Инициализировали локальную Spark-сессию;
- Прочитали DataFrame и кратко взглянули на его содержимое;
- Провели предобработку заключенных в нем данных, используя методы pySpark, а именно:
-- Исправили типы данных на более подходящие;
-- Исследовали данные на наличие пропусков;
-- Проверили данные на наличие дубликатов, которые не обнаружили;
-- Разбили данные данные на обучающую и тестовую выборки;
-- Заполнили пропуски медианой;
-- Преобразовали колонку с категориальными значениями;
-- Провели стандартизацию количественных признаков/

Затем мы построили две модели линейной регрессии на разных наборах данных:
- используя все данные из файла;
- используя только числовые переменные, исключив категориальные.

По итогу обучения моделей мы выяснили, что обе модели не обладают высокой точностью (возможно, моделям не хватает признаков, или признаки мало коррелируют с целевым, или выборка с данными слишком мала для получения качественных моделей). Но согласно всем трем метрикам RMSE, MAE и R2 точнее работает первая модель, использующая все данные из файла.

## Дополнительная работа: Альтернативный вариант работы с моделью (через пайплайн)

Пока мы делали все пошагово и вручную. Но попробуем построить процесс уже с помощью пайплайна и посмотреть, что выйдет.

Так как все преобразования данных (за исключением типов данных) мы проводили уже на разделеных а выборки данных, то заново датасет загружать для эксперимента нам не придется.

### Вариант 1

#### Подготовка пайплайна и обучение моделей

In [ ]:
# собираем вместе преобразования для работы с пайплайном
# заполнение пропусков
imputer = Imputer(strategy='median', inputCols=['total_bedrooms'], outputCols=['total_bedrooms'])
# категориальные признаки
indexer = StringIndexer(inputCols=categorical_cols,
                        outputCols=[c+'_idx' for c in categorical_cols],
                        handleInvalid = 'keep')
encoder = OHE(inputCols=[c+'_idx' for c in categorical_cols],
                        outputCols=[c+'_ohe' for c in categorical_cols])
categorical_assembler = \
        VectorAssembler(inputCols=[c+'_ohe' for c in categorical_cols],
                                        outputCol="categorical_features")
# числовые
numerical_assembler = VectorAssembler(inputCols=numerical_cols,
                      outputCol="numerical_features")
standardScaler = StandardScaler(inputCol='numerical_features',
                                outputCol="numerical_features_scaled")
# общая векторизация
all_features = ['categorical_features','numerical_features_scaled']

final_assembler = VectorAssembler(inputCols=all_features,
                                  outputCol="features")

In [ ]:
# порядок действий для первой модели
transformationsFP = [imputer, indexer, encoder, categorical_assembler,
                   numerical_assembler, standardScaler, final_assembler]

In [ ]:
# порядок действий для второй модели
transformationsSP = [imputer, numerical_assembler, standardScaler]

In [ ]:
#делим данные на обучающую и тестовую выборки (20% тестовая)
(trainingData, testData) = df_housing.randomSplit([0.8, 0.2], seed=RANDOM_SEED)
print(trainingData.count(), testData.count())

16418 4222


In [ ]:
# инициализируем первую модель
lrFP = LinearRegression(labelCol=target, # целевой признак
                         featuresCol='features', # используемые признаки
                         maxIter=10, # максимальное количество попыток, чтобы не уйти в бесконечность
                         regParam=0.3, # регуляризация раз
                         elasticNetParam=0.8) # регуляризация два
# обучаем первую модель
modelFP = Pipeline(stages=transformationsFP+[lrFP]).fit(trainingData)

In [ ]:
# инициализируем вторую модель
lrSP = LinearRegression(labelCol=target, # целевой признак
                         featuresCol='numerical_features_scaled', # используемые признаки
                         maxIter=10, # максимальное количество попыток, чтобы не уйти в бесконечность
                         regParam=0.3, # регуляризация раз
                         elasticNetParam=0.8) # регуляризация два
# обучаем вторую модель
modelSP = Pipeline(stages=transformationsSP+[lrSP]).fit(trainingData)

#### Анализ результатов

Оценим качество модели на обучающих данных.

In [ ]:
# сохраняем название колонки
predictions_col = 'prediction'

In [ ]:
# получаем предсказание
predictionsFP = modelFP.transform(testData)

In [ ]:
# сохраняем предсказания первой модели в отдельную таблицу
# для зрительной оценки
predictedLabesFP = predictionsFP.select(target, predictions_col)
predictedLabesFP.show()

+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|          103600.0|182381.42558303662|
|           50800.0|231958.15269474522|
|           58100.0|158055.72001596307|
|           68400.0|161069.30142661533|
|           72200.0|179694.00335049978|
|           67000.0|170284.80168316327|
|           81300.0|169438.12488770206|
|           70500.0|182212.77405659505|
|           60000.0|159802.82909377804|
|          109400.0|202052.46585005336|
|           74100.0|168073.51044610539|
|           74700.0|187418.53626101883|
|           90000.0|227896.53923914908|
|          104200.0| 219030.4506407592|
|           74100.0|175322.96557153808|
|           67500.0| 166635.5044951376|
|          103100.0|  71047.3646006973|
|           92500.0| 177508.4806094747|
|          128100.0|235160.54384411918|
|           99600.0| 204546.1429572152|
+------------------+------------------+
only showing top 20 rows



In [ ]:
evaluatorP = RegressionEvaluator(labelCol=target)

In [ ]:
# метрики первой модели
# для тестовых данных
print('Метрики первой модели (со всеми признаками) на тренировочных данных')
print(f'RMSE: {evaluatorP.evaluate(predictionsFP, {evaluatorP.metricName: "rmse"})}')
print(f'MAE: {evaluatorP.evaluate(predictionsFP, {evaluatorP.metricName: "mae"})}')
print(f'R2: {evaluatorP.evaluate(predictionsFP, {evaluatorP.metricName: "r2"})}')

Метрики первой модели (со всеми признаками) на тренировочных данных
RMSE: 68985.61921996386
MAE: 50088.73193415476
R2: 0.6484925508726122


In [ ]:
# получаем предсказание
predictionsSP = modelSP.transform(testData)

In [ ]:
# метрики второй модели
# для тестовых данных
print('Метрики второй модели (только с числовыми признаками) на тренировочных данных')
print(f'RMSE: {evaluatorP.evaluate(predictionsSP, {evaluatorP.metricName: "rmse"})}')
print(f'MAE: {evaluatorP.evaluate(predictionsSP, {evaluatorP.metricName: "mae"})}')
print(f'R2: {evaluatorP.evaluate(predictionsSP, {evaluatorP.metricName: "r2"})}')

Метрики второй модели (только с числовыми признаками) на тренировочных данных
RMSE: 69179.2353649093
MAE: 50874.007814179306
R2: 0.6465166891307079


***Еще один вариант подсчета метрики***

In [ ]:
# еще один вариант вывода метрики,
# с округлением и без предварительного предсказания
# для первой модели
print('Метрики первой модели (со всеми признаками) на тренировочных данных:')
print('RMSE:', round(evaluatorP.setParams(metricName="rmse").evaluate(modelFP.transform(testData)), 4))
print('MAE:', round(evaluatorP.setParams(metricName='mae').evaluate(modelFP.transform(testData)), 4))
print('R2:', round(evaluatorP.setParams(metricName='r2').evaluate(modelFP.transform(testData)), 4))

Метрики первой модели (со всеми признаками) на тренировочных данных:
RMSE: 68985.6192
MAE: 50088.7319
R2: 0.6485


In [ ]:
# еще один вариант вывода метрики,
# с округлением и без предварительного предсказания
# для второй модели
print('Метрики второй модели (только с числовыми признаками) на тренировочных данных:')
print('RMSE:', round(evaluatorP.setParams(metricName="rmse").evaluate(modelSP.transform(testData)), 4))
print('MAE:', round(evaluatorP.setParams(metricName='mae').evaluate(modelSP.transform(testData)), 4))
print('R2:', round(evaluatorP.setParams(metricName='r2').evaluate(modelSP.transform(testData)), 4))

Метрики второй модели (только с числовыми признаками) на тренировочных данных:
RMSE: 69179.2354
MAE: 50874.0078
R2: 0.6465


### Вариант 2

In [ ]:
# собираем вместе преобразования для работы с пайплайном
stages = []

# заполнение пропусков
imputer = Imputer(strategy='median', inputCols=['total_bedrooms'], outputCols=['total_bedrooms'])
stages += [imputer]

# преобразование категориальных колонок в бинарные вектора благодаря строковому преобразователю
stringIndexer = StringIndexer(inputCols=categorical_cols,
                              outputCols=[c+'_idx' for c in categorical_cols],
                              handleInvalid = 'keep')
encoder = OHE(inputCols=[c+'_idx' for c in categorical_cols],
                        outputCols=[c+'_ohe' for c in categorical_cols])
categoricalAssembler = VectorAssembler(inputCols=[c+'_ohe' for c in categorical_cols],
                                        outputCol="categorical_features")
stages += [stringIndexer, encoder, categoricalAssembler]

# зависит от численной колонки:
numericCols = numerical_cols
for numericCol in numericCols:
   numericalAssembler = VectorAssembler(inputCols=numericCols,\
                                        outputCol="numerical_features")
   standardScaler = StandardScaler(inputCol=numericalAssembler.getOutputCol(),\
                                   outputCol="numerical_features_scaled")
stages +=[numericalAssembler,standardScaler]

# общая векторизация
all_features = ['categorical_features','numerical_features_scaled']
final_assembler = VectorAssembler(inputCols=all_features,\
                                 outputCol="features")
stages += [final_assembler]

In [ ]:
#делим данные на обучающую и тестовую выборки (20% тестовая)
(trainingData, testData) = df_housing.randomSplit([0.8, 0.2],seed=RANDOM_SEED)
print(trainingData.count(), testData.count())

16418 4222


In [ ]:
# инициализируем первую модель
lrFP = LinearRegression(labelCol=target, # целевой признак
                         featuresCol='features', # используемые признаки
                         maxIter=10, # максимальное количество попыток, чтобы не уйти в бесконечность
                         regParam=0.3, # регуляризация раз
                         elasticNetParam=0.8) # регуляризация два
# обучаем первую модель
modelFP = Pipeline(stages=stages+[lrFP]).fit(trainingData)

In [ ]:
evaluatorP = RegressionEvaluator(labelCol=target)

In [ ]:
# еще один вариант вывода метрики,
# с округлением и без предварительного предсказания
# для первой модели
print('Метрики первой модели (со всеми признаками) на тренировочных данных:')
print('RMSE:', round(evaluatorP.setParams(metricName="rmse").evaluate(modelFP.transform(testData)), 4))
print('MAE:', round(evaluatorP.setParams(metricName='mae').evaluate(modelFP.transform(testData)), 4))
print('R2:', round(evaluatorP.setParams(metricName='r2').evaluate(modelFP.transform(testData)), 4))

Метрики первой модели (со всеми признаками) на тренировочных данных:
RMSE: 68985.6192
MAE: 50088.7319
R2: 0.6485


In [ ]:
# инициализируем вторую модель
lrSP = LinearRegression(labelCol=target, # целевой признак
                         featuresCol='numerical_features_scaled', # используемые признаки
                         maxIter=10, # максимальное количество попыток, чтобы не уйти в бесконечность
                         regParam=0.3, # регуляризация раз
                         elasticNetParam=0.8) # регуляризация два
# обучаем вторую модель
modelSP = Pipeline(stages=[imputer, numerical_assembler, standardScaler, lrSP]).fit(trainingData)

In [ ]:
# еще один вариант вывода метрики,
# с округлением и без предварительного предсказания
# для второй модели
print('Метрики второй модели (только с числовыми признаками) на тренировочных данных:')
print('RMSE:', round(evaluatorP.setParams(metricName="rmse").evaluate(modelSP.transform(testData)), 4))
print('MAE:', round(evaluatorP.setParams(metricName='mae').evaluate(modelSP.transform(testData)), 4))
print('R2:', round(evaluatorP.setParams(metricName='r2').evaluate(modelSP.transform(testData)), 4))

Метрики второй модели (только с числовыми признаками) на тренировочных данных:
RMSE: 69179.2354
MAE: 50874.0078
R2: 0.6465


In [ ]:
spark.stop()

Независимо от того, применялся ли пайплайн или нет, метрики у обеих моделей на тестовых данных не изменились, а значит и выводы - тоже.